In [1]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

In [2]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

Mlflow

In [10]:
from mlProject.constants.Mlflow_uri import *

[2026-06-26 15:37:38,044: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as kamounamal34

[2026-06-26 15:37:38,061: INFO: helpers: Accessing as kamounamal34]
[2026-06-26 15:37:38,753: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/kamounamal34/End-to-End-project-Mlops-Mlflow "HTTP/1.1 200 OK"]
[2026-06-26 15:37:39,397: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "kamounamal34/End-to-End-project-Mlops-Mlflow"

[2026-06-26 15:37:39,402: INFO: helpers: Initialized MLflow to track repo "kamounamal34/End-to-End-project-Mlops-Mlflow"]


Repository kamounamal34/End-to-End-project-Mlops-Mlflow initialized!

[2026-06-26 15:37:39,419: INFO: helpers: Repository kamounamal34/End-to-End-project-Mlops-Mlflow initialized!]


In [11]:
Mlflow_Tracking_uri 

'https://dagshub.com/kamounamal34/End-to-End-project-Mlops-Mlflow.mlflow'

Update the entity

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

Update the configuration manager

In [5]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = {k: v for k, v in  self.params.items() if k != "Model_name"}
        schema = self.schema.TARGET_COLUMN
        

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = Mlflow_Tracking_uri,
        )

        return model_evaluation_config

Update the components

In [7]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import numpy as np
import joblib
import glob

In [ ]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        params_filepath = PARAMS_FILE_PATH
        self.params = read_yaml(params_filepath)
        

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    

    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        
       # load the most recent model
        model_dir = self.config.model_path

        latest_model = max(
            glob.glob(os.path.join(model_dir, "*.joblib")),
            key=os.path.getmtime
        )

        model = joblib.load(latest_model)
        
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        # define the experiment name
        mlflow.set_experiment("Price_house_pred") 
        
        with mlflow.start_run(run_name = (self.params.Model_name +  "_model") ):

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # save metrics
            scores = {"rmse": rmse, "mae": mae, "r2": r2}


            # if the metric file name  already exists, add 1 to the metric file name to have a new metric file name 
            metric_file_name = self.config.metric_file_name  
            base_name, extension = os.path.splitext(metric_file_name)

            counter = 1
            new_metric_file_name = metric_file_name

            while os.path.exists(new_metric_file_name):
                new_metric_file_name = f"{base_name}{counter}{extension}"
                counter += 1

            save_json(path=Path(new_metric_file_name), data={**scores, 
                                                             "loaded_model_params": str(model),
                                                            "loaded_model_path": str(Path(latest_model).as_posix()) })

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            
            # if I am connected to an MLflow server (DagsHub) => register the model
            if tracking_url_type_store != "file":

                # register the model
                mlflow.sklearn.log_model(sk_model = model, artifact_path = "model", registered_model_name = self.params.Model_name)

            # if I am working locally, save the model as a file          
            else:  
                mlflow.sklearn.log_model(sk_model = model, artifact_path = "model", registered_model_name = self.params.Model_name)

Update the pipline

In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-06-26 15:37:44,465: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-26 15:37:44,469: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 15:37:44,477: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-26 15:37:44,481: INFO: common: created directory at: artifacts]
[2026-06-26 15:37:44,485: INFO: common: created directory at: artifacts/model_evaluation]
[2026-06-26 15:37:44,487: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 15:37:46,212: INFO: common: json file saved at: artifacts\model_evaluation\metrics1.json]


2026/06/26 15:37:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 15:38:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'ElasticNet' already exists. Creating a new version of this model...
2026/06/26 15:38:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet, version 18
Created version '18' of model 'ElasticNet'.


🏃 View run ElasticNet_model at: https://dagshub.com/kamounamal34/End-to-End-project-Mlops-Mlflow.mlflow/#/experiments/6/runs/bf17de4faaeb4955aa66691897014079
🧪 View experiment at: https://dagshub.com/kamounamal34/End-to-End-project-Mlops-Mlflow.mlflow/#/experiments/6


load model from Mlflow

In [19]:
import mlflow

In [20]:
from mlProject.constants.Mlflow_uri import *

In [22]:
# Dans Registered Models 
# Name : ElasticNet
# Version : 1
mlflow.sklearn.load_model(model_uri="models:/ElasticNet/1", dst_path="C:/Users/Amal/Desktop")

ElasticNet(alpha=0.2, l1_ratio=0.1, random_state=42)